# RAGScholar — Full Corpus RAG

**Goal:** build the complete academic corpus into one source-grounded RAG system with four retrieval lanes — **Text, Math, Visual, Table** — and use local **Ollama `llama3`** to synthesize the final answer from the **top 10 verified evidence items**.

This notebook is the multi-document successor to the single-document demo. It does not depend on the old experimental runtime, checkpoints, or document-specific caches. It discovers the PDFs under `data/`, builds each document independently, creates a corpus-level router, retrieves across the most relevant documents, reranks evidence globally, and finally asks `llama3` to answer only from retrieved evidence.

**Final flow**

`PDF corpus → MinerU → Canonical Objects → Text/Math/Visual/Table indexes → Corpus Router → Top Evidence → llama3 → Grounded Answer + Citations`

# 0. Configuration

All paths, model IDs, build parameters, corpus routing settings, and Ollama settings live here. The notebook discovers every PDF inside the configured corpus folders.

In [1]:
from pathlib import Path
from collections import defaultdict, OrderedDict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import gc, hashlib, json, os, re, shutil, sys, time, unicodedata

import numpy as np
import pandas as pd
import requests
import torch
import pymupdf
from bs4 import BeautifulSoup
from PIL import Image
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebook", "notebooks"} else cwd
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "ragscholar_full"
DOCUMENTS_ROOT = OUTPUT_ROOT / "documents"
CORPUS_ROOT = OUTPUT_ROOT / "corpus"

CORPUS_FOLDERS = ("Textbooks", "Survey", "Landmark", "Thesis", "Papers", "Research")
PARSE_TIER = "advanced"

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
RERANK_MODEL = "Qwen/Qwen3-Reranker-0.6B"
SIGLIP_MODEL = "google/siglip2-base-patch16-224"
OLLAMA_MODEL = "llama3"
OLLAMA_URL = "http://127.0.0.1:11434/api/chat"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SIGLIP_BUILD_DEVICE = DEVICE
SIGLIP_QUERY_DEVICE = "cpu"

RESET_FULL_BUILD = False
ENABLE_AUX_OCR = False
MAX_RUNTIME_DOCS = 1
ROUTER_TOP_DOCS = 5
TOP_FINAL_EVIDENCE = 10
GLOBAL_RERANK_POOL = 80
OLLAMA_GPU_FRIENDLY = True
OLLAMA_KEEP_ALIVE = 0

TEXT_CFG = {
    "target_tokens": 350, "soft_tokens": 420, "max_tokens": 500, "min_tokens": 80,
    "dense_k": 30, "bm25_k": 30, "rrf_k": 60, "rrf_top_k": 30,
    "anchor_budget": 5, "neighbor_budget": 4,
}
MATH_CFG = {
    "dense_k": 40, "bm25_k": 40, "source_chunk_k": 20, "source_equation_k": 30,
    "rrf_k": 60, "rerank_pool": 40, "evidence_k": 3,
}
VISUAL_CFG = {
    "candidate_k": 24, "evidence_k": 3, "rrf_k": 60, "rerank_pool": 10,
    "semantic_weights": {"qwen_source": 1.0, "bm25_source": 1.0, "ocr_lexical": 0.25},
    "visual_weights": {"qwen_source": 1.0, "siglip_multiview": 1.0, "ocr_lexical": 0.5, "bm25_source": 0.25},
}
TABLE_CFG = {
    "table_k": 16, "row_k": 24, "rrf_k": 60, "evidence_k": 3,
    "min_rows": 2, "min_cols": 2, "min_nonempty": 4, "min_density": 0.30,
}
ROUTER_CFG = {"dense_k": 15, "bm25_k": 15, "rrf_k": 60, "candidate_k": 12}

print("Project :", PROJECT_ROOT)
print("Data    :", DATA_ROOT)
print("Output  :", OUTPUT_ROOT)
print("Device  :", DEVICE)
print("Ollama  :", OLLAMA_MODEL)

Project : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt
Data    : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data
Output  : C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\ragscholar_full
Device  : cuda
Ollama  : llama3


## 0.1 Shared Utilities

Small helpers for hashing, JSON/JSONL persistence, text normalization, technical tokenization, rank fusion, and memory cleanup.

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(chunk_size), b""):
            h.update(part)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def write_jsonl(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def read_jsonl(path):
    path = Path(path)
    if not path.is_file(): return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def clean_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def short_text(value, n=1200):
    text = clean_text(value)
    return text if len(text) <= n else text[:n].rstrip() + "..."


def stable_hash(*parts, n=12):
    payload = json.dumps(parts, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:n]


TECH_TOKEN_RE = re.compile(r"[^\W\s]+(?:[./×+\-][^\W\s]+)*", re.UNICODE)
TECH_SPLIT_RE = re.compile(r"[./×+\-]")


def technical_split(text):
    out = []
    for token in TECH_TOKEN_RE.findall(str(text).lower()):
        out.append(token)
        if TECH_SPLIT_RE.search(token):
            out.extend(p for p in TECH_SPLIT_RE.split(token) if p and (len(p) >= 2 or p.isdigit()))
    return out


def weighted_rrf(lanes, weights, k=60, top_k=30, id_key="id"):
    scores, provenance = defaultdict(float), defaultdict(dict)
    for lane, rows in lanes.items():
        weight = float(weights.get(lane, 1.0))
        for row in rows:
            rid, rank = row[id_key], int(row["rank"])
            scores[rid] += weight / (k + rank)
            provenance[rid][lane] = {"rank": rank, "score": float(row.get("score", 0.0)), "weight": weight}
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))[:top_k]
    return [{id_key: rid, "rank": i, "score": float(score), "lanes": provenance[rid]}
            for i, (rid, score) in enumerate(ranked, 1)]


def package_version(name):
    try: return version(name)
    except PackageNotFoundError: return None


def release_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# 1. Corpus Discovery

The full corpus is discovered directly from `data/`. No old metadata notebook is required. Each PDF receives a deterministic document ID derived from its SHA-256 hash.

In [3]:
if RESET_FULL_BUILD and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

DOCUMENTS_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_RECORDS = []
for category in CORPUS_FOLDERS:
    folder = DATA_ROOT / category
    if not folder.is_dir():
        continue
    for path in sorted(folder.rglob("*.pdf")):
        sha = sha256_file(path)
        SOURCE_RECORDS.append({
            "document_id": f"doc_{sha[:12]}",
            "category": category,
            "filename": path.name,
            "source_path": str(path.resolve()),
            "source_sha256": sha,
        })

write_jsonl(CORPUS_ROOT / "source_registry.jsonl", SOURCE_RECORDS)
print("Documents discovered:", len(SOURCE_RECORDS))
display(pd.DataFrame(SOURCE_RECORDS)[["document_id", "category", "filename"]].head(20))

Documents discovered: 53


,document_id,category,filename
0,doc_89749245bc80,Textbooks,book1.pdf
1,doc_54d6bfa8cc4a,Textbooks,book2.pdf
2,doc_b3129f44c4a2,Textbooks,d2l-en.pdf
3,doc_8d098d65cf53,Textbooks,ESLII_print12_toc.pdf
4,doc_3f87b70c64a3,Textbooks,mml-book.pdf
5,doc_f8237d393163,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf
6,doc_585551fe1a17,Survey,survey_1.pdf
7,doc_d3058b36775f,Survey,survey_10.pdf
8,doc_4d4d323f9f7d,Survey,survey_11.pdf
9,doc_8991d9fd6227,Survey,survey_12.pdf


# 2. Retrieval Models

The Qwen embedding model is loaded once and reused for every document. The reranker and SigLIP models are loaded lazily so the batch build does not keep unnecessary models in GPU memory.

In [4]:
import faiss, bm25s
from bm25s.tokenization import Tokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModel, AutoProcessor

TEXT_TOKENIZER = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)
EMBEDDER = SentenceTransformer(EMBED_MODEL, device=DEVICE)
ENCODE_DOCS = EMBEDDER.encode_document if hasattr(EMBEDDER, "encode_document") else EMBEDDER.encode
RERANKER = None
SIGLIP_PROCESSOR = None
SIGLIP_MODEL_OBJ = None

TEXT_RERANK_PROMPT = "Rank academic passages by how directly and accurately they provide evidence for the question. Prefer explicit source evidence over merely related text."
MATH_RERANK_PROMPT = "Rank equations by whether the equation and its local source context satisfy the exact mathematical role requested. Distinguish definitions from related formulas and forward from inverse relations."


def get_reranker():
    global RERANKER
    if RERANKER is None:
        dtype = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16 if DEVICE == "cuda" else torch.float32
        RERANKER = CrossEncoder(
            RERANK_MODEL,
            device=DEVICE,
            model_kwargs={"torch_dtype": dtype},
            prompts={"research": TEXT_RERANK_PROMPT, "math_equation": MATH_RERANK_PROMPT},
            default_prompt_name="research",
        )
    return RERANKER


def get_siglip(device=None):
    global SIGLIP_PROCESSOR, SIGLIP_MODEL_OBJ
    device = device or SIGLIP_BUILD_DEVICE
    if SIGLIP_PROCESSOR is None:
        SIGLIP_PROCESSOR = AutoProcessor.from_pretrained(SIGLIP_MODEL)
    if SIGLIP_MODEL_OBJ is None:
        SIGLIP_MODEL_OBJ = AutoModel.from_pretrained(SIGLIP_MODEL).eval()
    SIGLIP_MODEL_OBJ = SIGLIP_MODEL_OBJ.to(device)
    return SIGLIP_PROCESSOR, SIGLIP_MODEL_OBJ


def encode_query(query):
    fn = EMBEDDER.encode_query if hasattr(EMBEDDER, "encode_query") else EMBEDDER.encode
    kwargs = {} if hasattr(EMBEDDER, "encode_query") else {"prompt_name": "query"}
    vec = fn([query], normalize_embeddings=True, convert_to_numpy=True, **kwargs).astype("float32")
    vec = np.ascontiguousarray(vec); faiss.normalize_L2(vec)
    return vec

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

# 3. MinerU Parsing

MinerU runs inside a worker thread to avoid the Jupyter event-loop conflict encountered by the synchronous MinerU API. Every document is parsed from the original PDF into its own artifact directory.

In [5]:
from mineru.parser import parse
from mineru.parser.writer import FileBasedDataWriter
from mineru.render import render, RenderFormat


def run_mineru(source_pdf, mineru_dir):
    mineru_dir = Path(mineru_dir); mineru_dir.mkdir(parents=True, exist_ok=True)

    def _parse():
        return parse(str(source_pdf), tier=PARSE_TIER, ocr_mode="auto", page_range="all")

    with ThreadPoolExecutor(max_workers=1) as executor:
        result = executor.submit(_parse).result()

    result.save(FileBasedDataWriter(str(mineru_dir)))
    content_pages = render(result.middle_json, RenderFormat.CONTENT_LIST_V2)
    return content_pages, result.structured_content()

# 4. Canonical Knowledge Document

MinerU output is converted into stable RAGScholar objects. Original PDF geometry, page identity, captions, equations, tables, and source relationships are preserved so all retrieval lanes share the same source of truth.

In [6]:
def spans_to_text(value):
    if value is None: return ""
    if isinstance(value, str): return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str): parts.append(item)
            elif isinstance(item, dict):
                t = str(item.get("type", "")).lower()
                c = spans_to_text(item.get("content", item.get("text", "")))
                parts.append(f"${c}$" if c and "equation" in t else c)
        return clean_text("".join(parts))
    if isinstance(value, dict): return spans_to_text(value.get("content", value.get("text", "")))
    return clean_text(value)


def canonical_type(mineru_type, payload):
    t = str(mineru_type or "").lower()
    if "title" in t: return "heading"
    if "paragraph" in t: return "paragraph"
    if "equation" in t: return "equation"
    if t == "image" or t.endswith("_image"): return "figure"
    if "chart" in t: return "chart"
    if "table" in t: return "table"
    if "algorithm" in t: return "algorithm"
    if "code" in t: return "code"
    if "index" in t: return "index"
    if "list" in t: return "reference" if "ref" in str(payload.get("list_type", "")).lower() else "list"
    if "page_header" in t or t == "header": return "header"
    if "page_footer" in t or t == "footer": return "footer"
    if "page_number" in t: return "page_number"
    if "footnote" in t: return "footnote"
    if "aside" in t: return "aside"
    return t or "unknown"


def block_text(block):
    t, c = str(block.get("type", "")).lower(), block.get("content") or {}
    if "title" in t: return spans_to_text(c.get("title_content"))
    if "paragraph" in t: return spans_to_text(c.get("paragraph_content"))
    if "equation" in t: return clean_text(c.get("math_content"))
    if t == "image": return spans_to_text(c.get("image_caption"))
    if "chart" in t: return spans_to_text(c.get("chart_caption"))
    if "table" in t: return spans_to_text(c.get("table_caption"))
    if "algorithm" in t: return spans_to_text(c.get("algorithm_content"))
    if "code" in t: return spans_to_text(c.get("code_content"))
    if "list" in t or "index" in t:
        return "\n".join(spans_to_text(x.get("item_content", x)) for x in c.get("list_items", []))
    for key, value in c.items():
        if key.endswith("_content"): return spans_to_text(value)
    return ""


def bbox_1000_to_pdf(bbox, page):
    if not bbox: return None
    x0, y0, x1, y1 = map(float, bbox)
    return [x0 * page["width_pt"] / 1000, y0 * page["height_pt"] / 1000,
            x1 * page["width_pt"] / 1000, y1 * page["height_pt"] / 1000]


def html_to_cells(html):
    if not html: return []
    table = BeautifulSoup(html, "lxml").find("table")
    if table is None: return []
    occupied, cells = set(), []
    for r, tr in enumerate(table.find_all("tr")):
        c = 0
        for cell in tr.find_all(["th", "td"], recursive=False):
            while (r, c) in occupied: c += 1
            rs, cs = int(cell.get("rowspan", 1) or 1), int(cell.get("colspan", 1) or 1)
            cells.append({"row": r, "col": c, "rowspan": rs, "colspan": cs,
                          "is_header": cell.name == "th", "text": clean_text(cell.get_text(" ", strip=True))})
            for rr in range(r, r + rs):
                for cc in range(c, c + cs): occupied.add((rr, cc))
            c += cs
    return cells

In [7]:
FIG_LABEL = re.compile(r"\b(?:fig(?:ure)?\.?)\s*(\d+(?:\.\d+)*)", re.I)
TABLE_LABEL = re.compile(r"\btable\s*(\d+(?:\.\d+)*)", re.I)
EQ_REF = re.compile(r"\b(?:eq(?:uation)?\.?)\s*(\d+(?:\.\d+)*)", re.I)
EQ_TAG = re.compile(r"\\tag\*?\s*\{\s*([^{}]+?)\s*\}", re.I)


def build_canonical_document(source_record, content_pages, structured_content, mineru_dir, pdf, dirs):
    document_id = source_record["document_id"]
    source_sha = source_record["source_sha256"]
    pages = [{
        "page_index": i, "pdf_page_number": i + 1, "printed_page_label": pdf[i].get_label() or None,
        "width_pt": float(pdf[i].rect.width), "height_pt": float(pdf[i].rect.height), "rotation": int(pdf[i].rotation),
    } for i in range(pdf.page_count)]
    toc = [{"level": int(level), "title": title.strip(), "page_index": max(int(page_no) - 1, 0)}
           for level, title, page_no in pdf.get_toc(simple=True)]

    def parser_asset(payload):
        raw = ((payload.get("image_source") or {}).get("path") or "").strip()
        if not raw: return None
        raw = raw.lstrip("/\\")
        direct, fallback = Path(mineru_dir) / raw, Path(mineru_dir) / "images" / Path(raw).name
        return str(direct if direct.exists() else fallback)

    def caption_footnote(kind, payload):
        keys = {
            "figure": ("image_caption", "image_footnote"), "chart": ("chart_caption", "chart_footnote"),
            "table": ("table_caption", "table_footnote"), "code": ("code_caption", "code_footnote"),
            "algorithm": ("algorithm_caption", "algorithm_footnote"),
        }
        a, b = keys.get(kind, (None, None))
        return (spans_to_text(payload.get(a)), spans_to_text(payload.get(b))) if a else (None, None)

    objects, section_titles, section_ids = [], [], []
    for page_idx, blocks in enumerate(content_pages[:len(pages)]):
        page = pages[page_idx]
        for order, block in enumerate(blocks):
            mineru_type, payload = block.get("type"), block.get("content") or {}
            kind, text, bbox_norm = canonical_type(mineru_type, payload), block_text(block), block.get("bbox")
            oid = f"{document_id}:p{page_idx:04d}:{kind}:{order:04d}:{stable_hash(kind, text, bbox_norm, n=8)}"

            if kind == "heading":
                level = max(int(payload.get("level", 0) or 1), 1)
                section_titles, section_ids = section_titles[:level - 1], section_ids[:level - 1]
                parent_heading_id = section_ids[-1] if section_ids else None
                section_path = section_titles + [text]
            else:
                parent_heading_id = section_ids[-1] if section_ids else None
                section_path = list(section_titles)

            caption, footnote = caption_footnote(kind, payload)
            html = payload.get("html") if kind == "table" else None
            objects.append({
                "object_id": oid, "document_id": document_id, "object_type": kind, "mineru_type": mineru_type,
                "page_index": page_idx, "pdf_page_number": page_idx + 1, "printed_page_label": page["printed_page_label"],
                "reading_order": order, "section_path": section_path, "parent_heading_id": parent_heading_id,
                "bbox_norm_1000": bbox_norm, "bbox_pdf": bbox_1000_to_pdf(bbox_norm, page), "content": text,
                "inline_spans": payload.get("paragraph_content") if kind == "paragraph" else payload.get("title_content") if kind == "heading" else None,
                "latex": payload.get("math_content") if kind == "equation" else None,
                "caption": caption, "footnote": footnote, "table_html": html, "table_cells": html_to_cells(html) if html else [],
                "asset_path": parser_asset(payload) if kind in {"figure", "chart", "table", "equation"} else None,
                "source_payload": payload, "parser": "mineru", "source_sha256": source_sha,
                "derived": False, "indexable": kind not in {"header", "footer", "page_number", "aside", "reference"},
            })
            if kind == "heading":
                section_titles.append(text); section_ids.append(oid)

    object_by_id = {o["object_id"]: o for o in objects}
    ordered = sorted(objects, key=lambda o: (o["page_index"], o["reading_order"]))

    labels = {}
    for obj in objects:
        label = None
        if obj["object_type"] in {"figure", "chart"}:
            m = FIG_LABEL.search(obj.get("caption") or ""); label = f"figure:{m.group(1)}" if m else None
        elif obj["object_type"] == "table":
            m = TABLE_LABEL.search(obj.get("caption") or ""); label = f"table:{m.group(1)}" if m else None
        elif obj["object_type"] == "equation":
            m = EQ_TAG.search(obj.get("latex") or ""); label = f"equation:{m.group(1)}" if m else None
        obj["label"] = label
        if label: labels[label.lower()] = obj["object_id"]

    for i, obj in enumerate(ordered):
        obj["previous_id"] = ordered[i - 1]["object_id"] if i else None
        obj["next_id"] = ordered[i + 1]["object_id"] if i + 1 < len(ordered) else None
        refs, text = [], obj.get("content") or ""
        for prefix, rx in (("figure", FIG_LABEL), ("table", TABLE_LABEL), ("equation", EQ_REF)):
            for number in rx.findall(text):
                target = labels.get(f"{prefix}:{number}".lower())
                if target and target != obj["object_id"]: refs.append(target)
        obj["references"] = list(dict.fromkeys(refs))

    page_objects = defaultdict(list)
    for obj in objects: page_objects[obj["page_index"]].append(obj["object_id"])
    final_pages = [dict(page, object_ids=page_objects[page["page_index"]]) for page in pages]
    source = {
        **source_record,
        "page_count": pdf.page_count,
        "pdf_metadata": pdf.metadata,
    }
    parser_meta = {
        "name": "mineru", "tier_requested": PARSE_TIER,
        "producer": structured_content.get("metadata", {}).get("producer", {}),
        "extensions": structured_content.get("extensions", {}),
    }
    knowledge_document = {
        "schema": "ragscholar.knowledge_document", "schema_version": "2.0",
        "source": source, "parser": parser_meta, "toc": toc, "pages": final_pages, "objects": objects,
    }
    write_json(dirs["canonical"] / "document.json", knowledge_document)
    write_jsonl(dirs["canonical"] / "objects.jsonl", objects)
    write_json(dirs["canonical"] / "labels.json", labels)
    return {"source": source, "pages": pages, "toc": toc, "objects": objects, "object_by_id": object_by_id, "ordered": ordered, "labels": labels}

# 5. Text RAG Builder

The text lane keeps the final demo design: structure-aware chunks → Qwen dense index + BM25 → RRF → Qwen reranking → source-bound evidence with local neighbor expansion.

In [8]:
TEXT_TYPES = {"heading", "paragraph", "list", "algorithm", "code", "footnote", "definition", "theorem", "proof", "worked_example"}
BRIDGE_TYPES = {"equation", "figure", "chart", "table"}
CHUNK_EXCLUDED_SECTIONS = {"index", "bibliography", "references"}
RETRIEVAL_EXCLUDED = {"index", "bibliography", "references", "contents", "table of contents", "attribution"}


def token_count(text):
    return len(TEXT_TOKENIZER.encode(text, add_special_tokens=False))


def split_long_text(text, limit):
    if token_count(text) <= limit: return [text]
    sentences, parts, buf = re.split(r"(?<=[.!?])\s+(?=\S)", text), [], ""
    for sentence in sentences:
        candidate = f"{buf} {sentence}".strip()
        if token_count(candidate) <= limit:
            buf = candidate
        else:
            if buf: parts.append(buf)
            if token_count(sentence) <= limit:
                buf = sentence
            else:
                ids = TEXT_TOKENIZER.encode(sentence, add_special_tokens=False)
                parts.extend(TEXT_TOKENIZER.decode(ids[i:i + limit], skip_special_tokens=True).strip() for i in range(0, len(ids), limit))
                buf = ""
    if buf: parts.append(buf)
    return [x for x in parts if x]


def build_text_rag(ctx, dirs):
    document_id, objects, ordered, object_by_id, pages = ctx["source"]["document_id"], ctx["objects"], ctx["ordered"], ctx["object_by_id"], ctx["pages"]
    text_objects = sorted([o for o in objects if o.get("indexable", True) and o["object_type"] in TEXT_TYPES and clean_text(o.get("content"))], key=lambda o: (o["page_index"], o["reading_order"]))
    text_ids = {o["object_id"] for o in text_objects}

    def excluded_section(path):
        return any(str(x).strip().lower() in CHUNK_EXCLUDED_SECTIONS for x in path or [])

    def retrieval_text(section, body):
        path = " > ".join(section)
        return f"Section: {path}\n\n{body}" if path else body

    heading_by_path = {tuple(o["section_path"]): o["object_id"] for o in text_objects if o["object_type"] == "heading" and not excluded_section(o["section_path"])}
    section_items = defaultdict(list)
    for obj in ordered:
        section = obj.get("section_path") or []
        if excluded_section(section) or obj["object_type"] == "heading": continue
        if obj["object_id"] in text_ids:
            prefix = token_count(retrieval_text(section, ""))
            limit = max(TEXT_CFG["min_tokens"], TEXT_CFG["max_tokens"] - prefix)
            for part_no, part in enumerate(split_long_text(obj["content"], limit)):
                section_items[tuple(section)].append({"kind": "text", "object_id": obj["object_id"], "part_no": part_no, "text": part, "page_index": obj["page_index"], "references": obj.get("references", [])})
        elif obj["object_type"] in BRIDGE_TYPES:
            label = (obj.get("label") or obj["object_type"]).replace(":", " ").title()
            section_items[tuple(section)].append({"kind": "bridge", "object_id": obj["object_id"], "text": f"[{label}]", "page_index": obj["page_index"], "references": []})

    def body(items, include_bridges=True):
        return "\n\n".join(x["text"] for x in items if x["kind"] == "text" or include_bridges)

    def make_chunk(section, items):
        text_items = [x for x in items if x["kind"] == "text"]
        source_ids = list(dict.fromkeys(x["object_id"] for x in text_items))
        bridge_ids = list(dict.fromkeys(x["object_id"] for x in items if x["kind"] == "bridge"))
        refs = list(dict.fromkeys(r for x in text_items for r in x["references"]))
        headings = [heading_by_path[tuple(section[:i])] for i in range(1, len(section) + 1) if tuple(section[:i]) in heading_by_path]
        source_text = body(items, False); search_text = retrieval_text(section, body(items, True))
        return {
            "chunk_id": f"{document_id}:text:{stable_hash(source_ids, bridge_ids, source_text, n=10)}",
            "document_id": document_id, "source_object_ids": source_ids, "bridge_object_ids": bridge_ids,
            "context_heading_ids": headings, "text": source_text, "retrieval_text": search_text,
            "section_path": list(section), "page_start": min(x["page_index"] for x in items), "page_end": max(x["page_index"] for x in items),
            "references": refs, "token_count": token_count(search_text),
        }

    chunks = []
    for section, items in section_items.items():
        current, groups = [], []
        for item in items:
            candidate = current + [item]
            candidate_tokens = token_count(retrieval_text(section, body(candidate)))
            current_tokens = token_count(retrieval_text(section, body(current))) if current else 0
            if current and any(x["kind"] == "text" for x in current) and (candidate_tokens > TEXT_CFG["max_tokens"] or (current_tokens >= TEXT_CFG["target_tokens"] and candidate_tokens > TEXT_CFG["soft_tokens"])):
                groups.append(current); current = [item]
            else:
                current = candidate
        if current and any(x["kind"] == "text" for x in current): groups.append(current)
        if len(groups) >= 2:
            last_tokens = token_count(retrieval_text(section, body(groups[-1])))
            merged_tokens = token_count(retrieval_text(section, body(groups[-2] + groups[-1])))
            if last_tokens < TEXT_CFG["min_tokens"] and merged_tokens <= TEXT_CFG["max_tokens"]:
                groups[-2] += groups[-1]; groups.pop()
        chunks.extend(make_chunk(section, group) for group in groups)

    body_pages = [p["page_index"] for p in pages if re.fullmatch(r"\d+", str(p.get("printed_page_label") or "").strip())]
    first_body_page = min(body_pages) if body_pages else 0

    def content_role(section):
        names = [str(x).strip().lower() for x in section]
        if "notes" in names: return "notes"
        if any(x.startswith("appendix") for x in names): return "appendix"
        if any(x in {"exercise", "exercises", "problem", "problems"} for x in names): return "exercise"
        return "main"

    records = []
    for row_id, chunk in enumerate(c for c in chunks if c["page_start"] >= first_body_page and not any(str(x).strip().lower() in RETRIEVAL_EXCLUDED for x in c.get("section_path", []))):
        section = chunk.get("section_path") or []
        bridge_terms = " ".join((object_by_id[oid].get("label") or object_by_id[oid]["object_type"]).replace(":", " ") for oid in chunk.get("bridge_object_ids", []) if oid in object_by_id)
        bm25_text = "\n".join(x for x in ((section[-1] if section else ""), chunk["text"], bridge_terms) if x)
        records.append({
            "row_id": row_id, "chunk_id": chunk["chunk_id"], "document_id": document_id,
            "source_text": chunk["text"], "dense_text": chunk["retrieval_text"], "bm25_text": bm25_text,
            "section_path": section, "content_role": content_role(section), "page_start": chunk["page_start"], "page_end": chunk["page_end"],
            "source_object_ids": chunk["source_object_ids"], "bridge_object_ids": chunk["bridge_object_ids"], "references": chunk["references"],
        })

    write_jsonl(dirs["text"] / "chunks.jsonl", chunks)
    write_jsonl(dirs["text"] / "records.jsonl", records)

    if records:
        vectors = ENCODE_DOCS([r["dense_text"] for r in records], batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
        vectors = np.ascontiguousarray(vectors); faiss.normalize_L2(vectors)
        dense = faiss.IndexFlatIP(vectors.shape[1]); dense.add(vectors)
        faiss.write_index(dense, str(dirs["text"] / "dense.faiss"))

        tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
        tokens = tok.tokenize([r["bm25_text"] for r in records], return_as="tuple")
        bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
        bm_dir = dirs["text"] / "bm25"; bm_dir.mkdir(parents=True, exist_ok=True)
        bm.save(str(bm_dir)); tok.save_vocab(save_dir=str(bm_dir)); tok.save_stopwords(save_dir=str(bm_dir))

    return {"chunks": chunks, "records": records}

# 6. Math RAG Builder

Equations are converted into source-linked equation cards with local context, formula roles, dense and lexical indexes, and links back to explanatory text chunks.

In [9]:
SOURCE_TAG_RE = re.compile(r"\\tag\*?\s*\{\s*([^{}]+?)\s*\}", re.I)
STYLE_RE = re.compile(r"\\(?:mathbf|mathrm|mathit|mathsf|mathtt|boldsymbol|bm)\s*\{([^{}]*)\}")
MATH_EXCLUDED_SECTIONS = {"index", "bibliography", "references", "contents", "table of contents", "attribution"}
MATH_CONTEXT_TYPES = {"paragraph", "list", "definition", "theorem", "proof", "worked_example", "algorithm", "footnote"}


def strip_equation_tag(latex):
    return SOURCE_TAG_RE.sub("", unicodedata.normalize("NFKC", str(latex or ""))).strip()


def normalize_formula(latex):
    text = strip_equation_tag(latex)
    for token in ("\\left", "\\right", "\\,", "\\;", "\\!", "\\quad", "\\qquad"): text = text.replace(token, "")
    for _ in range(4):
        newer = STYLE_RE.sub(r"\1", text)
        if newer == text: break
        text = newer
    return re.sub(r"\s+", "", text).strip()


def formula_roles(card):
    latex = str(card.get("original_latex") or "")
    compact = re.sub(r"\s+", "", strip_equation_tag(latex)); lower = compact.lower()
    context = (" ".join(card.get("source_contexts", [])) + " " + " > ".join(card.get("section_path", []))).lower()
    lhs = compact.split("=", 1)[0] if "=" in compact else compact
    roles = set()
    has_softmax = "softmax" in lower
    has_sigmoid = "sigmoid" in lower or "sig[" in lower or "sig(" in lower
    has_probability = bool(re.search(r"(?:\\Pr|Pr|P\s*r)\s*\(", latex))
    has_partial = r"\partial" in latex
    has_det = bool(re.search(r"\\det\b", latex, re.I) or (has_partial and (r"\left|" in latex or re.search(r"\|\s*\\frac\s*\{\s*\\partial", latex))))
    has_inverse = bool(re.search(r"\^\s*\{\s*-\s*1\s*\}", latex))
    has_log, has_sum = r"\log" in latex, r"\sum" in latex
    has_kl = bool(re.search(r"\bKL\b", latex, re.I))
    has_argmin, has_sqrt = "argmin" in lower, r"\sqrt" in latex
    is_loss = bool(re.search(r"(?:^|[^A-Za-z])L\s*[\[\(\{]", latex) or has_argmin or (has_sum and has_log and ("negative log" in context or "loss" in context)))
    if has_softmax: roles.add("softmax")
    if has_sigmoid: roles.add("sigmoid")
    if has_probability: roles.add("probability")
    if is_loss: roles.add("loss")
    if has_softmax and not has_probability and not is_loss and r"\exp" in latex: roles.add("standalone_softmax")
    if re.search(r"\\(?:mathbf|boldsymbol)\{k\}|(?:^|[^A-Za-z])k[_\{\[]", lhs): roles.add("key_projection")
    if re.search(r"(?:^|[^A-Za-z])a(?:_|\[|\{)", lhs) and (has_softmax or r"\exp" in latex): roles.add("attention_weights")
    if has_sum and ("value vector" in context or "weighted sum" in context): roles.add("weighted_value_sum")
    if has_sqrt and (has_softmax or r"\exp" in latex) and "attention" in context: roles.add("scaled_attention")
    if "matrix form" in context and "attention" in context: roles.add("matrix_attention")
    if has_partial and has_det: roles.add("jacobian_determinant")
    elif has_partial and not has_log and not has_kl: roles.add("jacobian_matrix")
    simple_mapping = "=" in compact and not any((has_partial, has_probability, has_log, has_kl, has_argmin))
    forward_shape = bool(re.search(r"(?:\\mathbf\{)?x(?:\})?=.*(?:\\mathbf\{)?f(?:\})?.*(?:\\mathbf\{)?z(?:\})?", compact, re.I))
    if simple_mapping and forward_shape and not has_inverse: roles.add("forward_mapping")
    if simple_mapping and has_inverse: roles.add("inverse_mapping")
    if "coupling" in context and simple_mapping: roles.add("coupling_inverse" if has_inverse else "coupling_forward")
    if has_probability and has_partial and has_det: roles.add("change_of_variables_density")
    if has_kl and (has_argmin or is_loss): roles.add("kl_objective")
    if has_log and has_det and not has_kl and (has_argmin or is_loss or "maximum likelihood" in context): roles.add("logdet_objective")
    return sorted(roles)

In [10]:
def build_math_rag(ctx, text_data, dirs):
    document_id, objects, object_by_id = ctx["source"]["document_id"], ctx["objects"], ctx["object_by_id"]
    text_records = text_data["records"]
    text_by_chunk = {r["chunk_id"]: r for r in text_records}

    def allowed(obj):
        return not any(str(x).strip().lower() in MATH_EXCLUDED_SECTIONS for x in obj.get("section_path") or [])

    equations = sorted([o for o in objects if o["object_type"] == "equation" and o.get("indexable", True) and clean_text(o.get("latex")) and allowed(o)], key=lambda o: (o["page_index"], o["reading_order"]))
    records = []
    for row_id, eq in enumerate(equations):
        records.append({
            "row_id": row_id, "object_id": eq["object_id"], "document_id": document_id,
            "equation_label": eq.get("label"), "original_latex": eq["latex"], "normalized_formula": normalize_formula(eq["latex"]),
            "section_path": list(eq.get("section_path") or []), "page_index": eq["page_index"], "pdf_page_number": eq["pdf_page_number"],
            "printed_page_label": eq.get("printed_page_label"), "reading_order": eq["reading_order"], "bbox_pdf": eq.get("bbox_pdf"),
            "previous_id": eq.get("previous_id"), "next_id": eq.get("next_id"),
        })
    by_object = {r["object_id"]: r for r in records}
    by_label = {r["equation_label"].lower(): r for r in records if r.get("equation_label")}

    eq_number_re = re.compile(r"[A-Za-z0-9]+(?:\.[A-Za-z0-9]+)+")
    edges, seen = [], set()

    def add_edge(chunk, equation_object_id, edge_type, weight):
        key = (chunk["chunk_id"], equation_object_id, edge_type)
        if equation_object_id in by_object and key not in seen:
            seen.add(key); edges.append({"chunk_id": chunk["chunk_id"], "equation_object_id": equation_object_id, "edge_type": edge_type, "weight": weight})

    for chunk in text_records:
        for oid in chunk.get("references", []): add_edge(chunk, oid, "explicit_object_reference", 1.0)
        for number in eq_number_re.findall(str(chunk.get("source_text", ""))):
            key = f"equation:{number}".lower()
            if key in by_label: add_edge(chunk, by_label[key]["object_id"], "textual_equation_reference", 0.98)
        for oid in chunk.get("bridge_object_ids", []): add_edge(chunk, oid, "direct_bridge", 0.85)

    eq_to_source = defaultdict(list)
    for edge in edges: eq_to_source[edge["equation_object_id"]].append(edge)

    def local_context(record, max_objects=4):
        origin, section = object_by_id[record["object_id"]], tuple(record.get("section_path") or [])
        contexts = []
        for pointer in ("previous_id", "next_id"):
            oid, hops = origin.get(pointer), 0
            while oid and hops < 12 and len(contexts) < max_objects:
                obj = object_by_id.get(oid)
                if not obj or tuple(obj.get("section_path") or []) != section or obj["object_type"] == "equation": break
                if obj["object_type"] in MATH_CONTEXT_TYPES and clean_text(obj.get("content")): contexts.append(clean_text(obj["content"]))
                oid, hops = obj.get(pointer), hops + 1
        for edge in eq_to_source.get(record["object_id"], [])[:3]:
            chunk = text_by_chunk.get(edge["chunk_id"])
            if chunk: contexts.append(clean_text(chunk["source_text"]))
        return list(dict.fromkeys(contexts))[:6]

    cards = []
    for record in records:
        contexts = local_context(record); section = " > ".join(record.get("section_path") or [])
        semantic = "\n".join(x for x in (f"Section: {section}" if section else "", *contexts[:4], strip_equation_tag(record["original_latex"])) if x)
        lexical = "\n".join(x for x in ((record.get("equation_label") or "").replace(":", " "), section, *contexts, strip_equation_tag(record["original_latex"])) if x)
        card = dict(record, source_contexts=contexts, semantic_text=semantic, lexical_text=lexical)
        card["formula_roles"] = formula_roles(card); cards.append(card)

    write_jsonl(dirs["math"] / "equations.jsonl", records)
    write_jsonl(dirs["math"] / "source_graph.jsonl", edges)
    write_jsonl(dirs["math"] / "equation_cards.jsonl", cards)

    if cards:
        vectors = EMBEDDER.encode([c["semantic_text"] for c in cards], batch_size=16, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
        vectors = np.ascontiguousarray(vectors); faiss.normalize_L2(vectors)
        dense = faiss.IndexFlatIP(vectors.shape[1]); dense.add(vectors); faiss.write_index(dense, str(dirs["math"] / "dense.faiss"))
        tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
        tokens = tok.tokenize([c["lexical_text"] for c in cards], return_as="tuple")
        bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
        bm_dir = dirs["math"] / "bm25"; bm_dir.mkdir(parents=True, exist_ok=True)
        bm.save(str(bm_dir)); tok.save_vocab(save_dir=str(bm_dir)); tok.save_stopwords(save_dir=str(bm_dir))

    return {"records": records, "cards": cards, "source_graph": edges}

# 7. Visual RAG Builder

The visual lane keeps the final routed design: source-bound crops + caption/context Qwen retrieval + BM25 + optional OCR + multi-view SigLIP2. OCR is auxiliary; the original crop and canonical caption remain authoritative.

In [11]:
VISUAL_TYPES = {"figure", "chart"}


def visual_views(image):
    w, h = image.size
    views = [("full", [0, 0, w, h], image)]
    if min(w, h) >= 256:
        cw, ch = max(w // 2, 128), max(h // 2, 128)
        boxes = [("top_left", [0, 0, cw, ch]), ("top_right", [w-cw, 0, w, ch]),
                 ("bottom_left", [0, h-ch, cw, h]), ("bottom_right", [w-cw, h-ch, w, h])]
        views.extend((name, box, image.crop(tuple(box))) for name, box in boxes)
    return views


def build_visual_rag(ctx, pdf, dirs):
    document_id, objects, ordered = ctx["source"]["document_id"], ctx["objects"], ctx["ordered"]
    visual_objects = sorted([o for o in objects if o["object_type"] in VISUAL_TYPES and o.get("bbox_pdf")], key=lambda o: (o["page_index"], o["reading_order"]))
    crops_dir = dirs["visual"] / "crops"; crops_dir.mkdir(parents=True, exist_ok=True)

    def nearby_text(obj, radius=2):
        same_page = [o for o in ordered if o["page_index"] == obj["page_index"]]
        pos = next((i for i, o in enumerate(same_page) if o["object_id"] == obj["object_id"]), None)
        if pos is None: return ""
        nearby = []
        for other in same_page[max(0, pos-radius):pos+radius+1]:
            if other["object_id"] != obj["object_id"] and other["object_type"] in TEXT_TYPES and clean_text(other.get("content")):
                nearby.append(clean_text(other["content"]))
        return " ".join(nearby)

    records = []
    for row_id, obj in enumerate(visual_objects):
        page = pdf.load_page(obj["page_index"])
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2.0, 2.0), clip=pymupdf.Rect(obj["bbox_pdf"]), alpha=False)
        crop_path = crops_dir / f"{stable_hash(obj['object_id'], n=16)}.png"; pix.save(str(crop_path))
        words = page.get_text("words", clip=pymupdf.Rect(obj["bbox_pdf"]))
        visible_text = clean_text(" ".join(w[4] for w in sorted(words, key=lambda w: (w[1], w[0]))))
        label = obj.get("label"); reference_key = label.split(":", 1)[1].lower() if label and ":" in label else None
        source_text = "\n".join(x for x in (obj.get("caption") or "", nearby_text(obj)) if x)
        records.append({
            "row_id": row_id, "visual_id": obj["object_id"], "document_id": document_id, "visual_type": obj["object_type"],
            "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or [], "bbox_pdf": obj["bbox_pdf"],
            "label": label, "reference_key": reference_key, "caption": obj.get("caption"), "source_text": source_text,
            "visible_text": visible_text, "ocr_text": "", "crop_path": str(crop_path), "crop_sha256": sha256_file(crop_path),
            "visual_group_id": label.lower() if label else None,
        })

    if ENABLE_AUX_OCR and records:
        import easyocr
        reader = easyocr.Reader(["en"], gpu=(DEVICE == "cuda"))
        for row in records:
            if len(row["visible_text"]) < 30:
                row["ocr_text"] = clean_text(" ".join(reader.readtext(row["crop_path"], detail=0)))

    write_jsonl(dirs["visual"] / "records.jsonl", records)
    if not records: return {"records": [], "view_meta": []}

    visual_text = ["\n".join(x for x in (r.get("label") or "", " > ".join(r["section_path"]), r["source_text"], r["visible_text"]) if x) for r in records]
    vectors = ENCODE_DOCS(visual_text, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    vectors = np.ascontiguousarray(vectors); faiss.normalize_L2(vectors)
    qwen_index = faiss.IndexFlatIP(vectors.shape[1]); qwen_index.add(vectors); faiss.write_index(qwen_index, str(dirs["visual"] / "qwen.faiss"))

    tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
    tokens = tok.tokenize(visual_text, return_as="tuple")
    bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
    bm_dir = dirs["visual"] / "bm25"; bm_dir.mkdir(parents=True, exist_ok=True)
    bm.save(str(bm_dir)); tok.save_vocab(save_dir=str(bm_dir)); tok.save_stopwords(save_dir=str(bm_dir))

    if any(clean_text(r.get("ocr_text")) for r in records):
        ocr_tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
        ocr_tokens = ocr_tok.tokenize([r.get("ocr_text") or "" for r in records], return_as="tuple")
        ocr_bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); ocr_bm.index(ocr_tokens)
        ocr_dir = dirs["visual"] / "ocr_bm25"; ocr_dir.mkdir(parents=True, exist_ok=True)
        ocr_bm.save(str(ocr_dir)); ocr_tok.save_vocab(save_dir=str(ocr_dir)); ocr_tok.save_stopwords(save_dir=str(ocr_dir))

    processor, model = get_siglip(SIGLIP_BUILD_DEVICE)
    view_meta, view_vectors, batch_images, batch_meta = [], [], [], []

    def flush():
        nonlocal batch_images, batch_meta
        if not batch_images: return
        inputs = processor(images=batch_images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(SIGLIP_BUILD_DEVICE)
        with torch.inference_mode():
            feats = model.get_image_features(pixel_values=pixel_values)
            if hasattr(feats, "pooler_output"): feats = feats.pooler_output
            feats = torch.nn.functional.normalize(feats.float(), dim=-1)
        view_vectors.append(feats.cpu().numpy().astype("float32")); view_meta.extend(batch_meta)
        batch_images, batch_meta = [], []

    for row in records:
        with Image.open(row["crop_path"]) as im:
            rgb = im.convert("RGB")
            for name, box, view in visual_views(rgb):
                batch_images.append(view); batch_meta.append({"visual_id": row["visual_id"], "view_type": name, "bbox_px": box})
                if len(batch_images) >= (20 if DEVICE == "cuda" else 4): flush()
    flush()

    matrix = np.concatenate(view_vectors, axis=0) if view_vectors else np.empty((0, 0), dtype="float32")
    np.save(dirs["visual"] / "siglip_multiview.npy", matrix)
    write_jsonl(dirs["visual"] / "siglip_multiview_meta.jsonl", view_meta)
    return {"records": records, "view_meta": view_meta}

# 8. Table RAG Builder

Tables are reconstructed into accepted table records, row records, and typed cell records. Table-level and row-level dense/BM25 indexes are built independently.

In [12]:
def numeric_value(text):
    raw = clean_text(text).replace(",", "")
    if not raw: return None
    m = re.fullmatch(r"([+-]?\d+(?:\.\d+)?)\s*(%|[A-Za-zµμ/$]+)?", raw)
    if not m: return None
    return {"value": float(m.group(1)), "unit": m.group(2) or None, "raw": clean_text(text)}


def expand_table_cells(cells):
    expanded = []
    for cell in cells:
        for r in range(cell["row"], cell["row"] + cell.get("rowspan", 1)):
            for c in range(cell["col"], cell["col"] + cell.get("colspan", 1)):
                expanded.append({"row": r, "col": c, "text": cell["text"], "is_header": cell.get("is_header", False), "origin_row": cell["row"], "origin_col": cell["col"]})
    return expanded


def table_quality(cells):
    if not cells: return {"accepted": False, "reason": "no_cells", "rows": 0, "cols": 0, "density": 0.0}
    rows = max(c["row"] for c in cells) + 1; cols = max(c["col"] for c in cells) + 1
    nonempty = sum(bool(clean_text(c.get("text"))) for c in cells); density = nonempty / max(rows * cols, 1)
    accepted = rows >= TABLE_CFG["min_rows"] and cols >= TABLE_CFG["min_cols"] and nonempty >= TABLE_CFG["min_nonempty"] and density >= TABLE_CFG["min_density"]
    return {"accepted": accepted, "reason": "accepted" if accepted else "insufficient_structure", "rows": rows, "cols": cols, "nonempty": nonempty, "density": round(density, 4)}


def build_hybrid_index(texts, out_dir, prefix):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    if not texts: return
    vectors = ENCODE_DOCS(texts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    vectors = np.ascontiguousarray(vectors); faiss.normalize_L2(vectors)
    dense = faiss.IndexFlatIP(vectors.shape[1]); dense.add(vectors); faiss.write_index(dense, str(out_dir / f"{prefix}.faiss"))
    tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None); tokens = tok.tokenize(texts, return_as="tuple")
    bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
    bm_dir = out_dir / f"{prefix}_bm25"; bm_dir.mkdir(parents=True, exist_ok=True)
    bm.save(str(bm_dir)); tok.save_vocab(save_dir=str(bm_dir)); tok.save_stopwords(save_dir=str(bm_dir))


def build_table_rag(ctx, pdf, dirs):
    objects = ctx["objects"]
    table_objects = sorted([o for o in objects if o["object_type"] == "table" and o.get("bbox_pdf")], key=lambda o: (o["page_index"], o["reading_order"]))
    tables, cells_out, rows_out, candidates = [], [], [], []

    def pymupdf_cells(obj):
        try:
            page = pdf.load_page(obj["page_index"]); finder = page.find_tables(clip=pymupdf.Rect(obj["bbox_pdf"]))
            if not finder.tables: return []
            data = finder.tables[0].extract()
            return [{"row": r, "col": c, "text": clean_text(value), "is_header": r == 0, "origin_row": r, "origin_col": c}
                    for r, row in enumerate(data) for c, value in enumerate(row) if clean_text(value)]
        except Exception:
            return []

    for table_idx, obj in enumerate(table_objects):
        expanded = expand_table_cells(obj.get("table_cells") or []) or pymupdf_cells(obj)
        quality = table_quality(expanded)
        candidates.append({"table_id": obj["object_id"], "pdf_page_number": obj["pdf_page_number"], "label": obj.get("label"), "caption": obj.get("caption"), **quality})
        if not quality["accepted"]: continue

        max_row, max_col = max(c["row"] for c in expanded), max(c["col"] for c in expanded)
        cell_map = {(c["row"], c["col"]): c for c in expanded}
        headers = {c: cell_map.get((0, c), {}).get("text", "") for c in range(max_col + 1)}
        table_id, table_cells, row_ids = obj["object_id"], [], []

        for c in expanded:
            cid = f"{table_id}:r{c['row']:04d}:c{c['col']:04d}"
            record = {"cell_id": cid, "table_id": table_id, "row": c["row"], "col": c["col"], "text": c["text"],
                      "header": headers.get(c["col"]) or None, "is_header": c["is_header"], "typed_value": numeric_value(c["text"]),
                      "pdf_page_number": obj["pdf_page_number"], "bbox_pdf": obj["bbox_pdf"]}
            cells_out.append(record); table_cells.append(record)

        for r in range(max_row + 1):
            values = [cell_map.get((r, c), {}).get("text", "") for c in range(max_col + 1)]
            row_text = " | ".join(x for x in values if x)
            if not row_text: continue
            rid = f"{table_id}:row:{r:04d}"; row_ids.append(rid)
            rows_out.append({"row_id": rid, "table_id": table_id, "row": r, "text": row_text, "values": values,
                             "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or []})

        caption = obj.get("caption") or obj.get("content") or ""
        table_text = "\n".join([caption, " > ".join(obj.get("section_path") or []), *[r["text"] for r in rows_out if r["table_id"] == table_id]])
        tables.append({"table_id": table_id, "row_id": table_idx, "label": obj.get("label"), "caption": caption,
                       "pdf_page_number": obj["pdf_page_number"], "section_path": obj.get("section_path") or [], "bbox_pdf": obj["bbox_pdf"],
                       "table_text": table_text, "cell_ids": [c["cell_id"] for c in table_cells], "row_ids": row_ids, "reconstruction": "structured"})

    write_jsonl(dirs["table"] / "candidates.jsonl", candidates)
    write_jsonl(dirs["table"] / "tables.jsonl", tables)
    write_jsonl(dirs["table"] / "rows.jsonl", rows_out)
    write_jsonl(dirs["table"] / "cells.jsonl", cells_out)
    build_hybrid_index([t["table_text"] for t in tables], dirs["table"], "tables")
    build_hybrid_index([r["text"] for r in rows_out], dirs["table"], "rows")
    return {"tables": tables, "rows": rows_out, "cells": cells_out, "candidates": candidates}

# 9. Per-Document Build

Each PDF is processed independently and written under `artifacts/ragscholar_full/documents/{document_id}`. A failed document is logged and the batch continues with the remaining corpus.

In [13]:
def document_dirs(document_id):
    root = DOCUMENTS_ROOT / document_id
    dirs = {name: root / name for name in ("parse", "canonical", "text", "math", "visual", "table")}
    for path in dirs.values(): path.mkdir(parents=True, exist_ok=True)
    return root, dirs


def collect_artifacts(root, exclude={"manifest.json"}):
    rows = []
    for path in sorted(Path(root).rglob("*")):
        rel = str(path.relative_to(root)).replace("\\", "/")
        if path.is_file() and rel not in exclude:
            rows.append({"path": rel, "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
    return rows


def build_document(source_record):
    document_id = source_record["document_id"]
    source_pdf = Path(source_record["source_path"])
    root, dirs = document_dirs(document_id)
    mineru_dir = dirs["parse"] / "mineru"

    pdf = pymupdf.open(source_pdf)
    try:
        content_pages, structured = run_mineru(source_pdf, mineru_dir)
        ctx = build_canonical_document(source_record, content_pages, structured, mineru_dir, pdf, dirs)
        text_data = build_text_rag(ctx, dirs)
        math_data = build_math_rag(ctx, text_data, dirs)
        visual_data = build_visual_rag(ctx, pdf, dirs)
        table_data = build_table_rag(ctx, pdf, dirs)

        title = clean_text((ctx["source"].get("pdf_metadata") or {}).get("title")) or source_pdf.stem
        headings = [clean_text(o.get("content")) for o in ctx["objects"] if o["object_type"] == "heading" and clean_text(o.get("content"))][:40]
        preview = " ".join(r["source_text"] for r in text_data["records"][:8])
        router_text = "\n".join(x for x in [title, source_record["category"], " | ".join(headings), short_text(preview, 4500)] if x)

        manifest = {
            "schema": "ragscholar.document_manifest_v2",
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "document_id": document_id, "category": source_record["category"], "source": ctx["source"],
            "title": title, "router_text": router_text,
            "models": {"embedding": EMBED_MODEL, "reranker": RERANK_MODEL, "siglip": SIGLIP_MODEL},
            "counts": {
                "pages": len(ctx["pages"]), "objects": len(ctx["objects"]), "text_records": len(text_data["records"]),
                "equations": len(math_data["cards"]), "visuals": len(visual_data["records"]),
                "table_candidates": len(table_data["candidates"]), "tables": len(table_data["tables"]),
                "table_rows": len(table_data["rows"]), "table_cells": len(table_data["cells"]),
            },
        }
        write_json(root / "manifest.json", manifest)
        manifest["artifacts"] = collect_artifacts(root)
        write_json(root / "manifest.json", manifest)
        return {"status": "built", "document_id": document_id, "category": source_record["category"], "title": title,
                "source_path": str(source_pdf), "router_text": router_text, "counts": manifest["counts"]}
    finally:
        pdf.close(); release_memory()

# 10. Full Corpus Build

This is the main batch execution. Every discovered document is built with the same four-modality pipeline.

In [14]:
import pickle, subprocess, textwrap

TEXTBOOKS_PER_RUN = 1
OTHER_PAGE_BUDGET = 600
OTHER_MAX_DOCS = 12
SKIP_DOCS = set()

BUILD_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_base_run_mineru = run_mineru
_base_get_siglip = get_siglip
_base_build_visual_rag = build_visual_rag
_base_encode_docs = ENCODE_DOCS
_old_max_seq = getattr(EMBEDDER, "max_seq_length", None)

if _old_max_seq is None or _old_max_seq > 1024:
    EMBEDDER.max_seq_length = 1024


def completed(source):
    path = DOCUMENTS_ROOT / source["document_id"] / "manifest.json"
    if not path.is_file():
        return False
    try:
        manifest = json.loads(path.read_text(encoding="utf-8"))
        return (
            manifest.get("document_id") == source["document_id"]
            and manifest.get("source", {}).get("source_sha256") == source["source_sha256"]
        )
    except Exception:
        return False


def page_count(source):
    with pymupdf.open(source["source_path"]) as pdf:
        return pdf.page_count


def clean_retry(source):
    root = DOCUMENTS_ROOT / source["document_id"]
    if not root.exists():
        return

    for name in ("canonical", "text", "math", "visual", "table"):
        path = root / name
        if path.exists():
            shutil.rmtree(path)

    manifest = root / "manifest.json"
    if manifest.exists():
        manifest.unlink()


MINERU_WORKER = textwrap.dedent(r"""
import pickle, sys
from pathlib import Path
from mineru.parser import parse
from mineru.parser.writer import FileBasedDataWriter
from mineru.render import render, RenderFormat

source_pdf, out_dir, tier = Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3]
out_dir.mkdir(parents=True, exist_ok=True)

result = parse(str(source_pdf), tier=tier, ocr_mode="auto", page_range="all")
result.save(FileBasedDataWriter(str(out_dir)))

payload = (
    render(result.middle_json, RenderFormat.CONTENT_LIST_V2),
    result.structured_content(),
)

with (out_dir / "_ragscholar_parse.pkl").open("wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
""")


def run_mineru(source_pdf, mineru_dir):
    mineru_dir = Path(mineru_dir)
    cache = mineru_dir / "_ragscholar_parse.pkl"

    if cache.is_file():
        print(" MinerU: using saved parse")
        with cache.open("rb") as f:
            return pickle.load(f)

    if mineru_dir.exists():
        shutil.rmtree(mineru_dir)

    EMBEDDER.to("cpu")
    release_memory()

    try:
        subprocess.run(
            [
                sys.executable, "-c", MINERU_WORKER,
                str(source_pdf), str(mineru_dir), PARSE_TIER,
            ],
            check=True,
        )
    finally:
        EMBEDDER.to(BUILD_DEVICE)
        release_memory()

    with cache.open("rb") as f:
        return pickle.load(f)


def safe_encode_docs(texts, batch_size=8, **kwargs):
    sizes = []
    for bs in (min(batch_size, 8), 4, 2, 1):
        if bs > 0 and bs not in sizes:
            sizes.append(bs)

    last_error = None

    for bs in sizes:
        try:
            return _base_encode_docs(texts, batch_size=bs, **kwargs)
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            last_error = exc
            release_memory()
            print(f" CUDA OOM → retrying embedding with batch_size={max(bs // 2, 1)}")

    raise last_error


ENCODE_DOCS = safe_encode_docs


def get_siglip(device=None):
    if BUILD_DEVICE == "cuda":
        EMBEDDER.to("cpu")
        release_memory()

    return _base_get_siglip(device)


def build_visual_rag(ctx, pdf, dirs):
    global SIGLIP_MODEL_OBJ

    result = _base_build_visual_rag(ctx, pdf, dirs)

    if SIGLIP_MODEL_OBJ is not None:
        SIGLIP_MODEL_OBJ.to("cpu")
        SIGLIP_MODEL_OBJ = None

    release_memory()
    EMBEDDER.to(BUILD_DEVICE)
    release_memory()

    return result


def clear_document_memory():
    global RERANKER, SIGLIP_MODEL_OBJ, SIGLIP_PROCESSOR

    if "RUNTIME_POOL" in globals():
        RUNTIME_POOL.clear()

    if RERANKER is not None:
        try:
            RERANKER.model.to("cpu")
        except Exception:
            pass
        RERANKER = None

    if SIGLIP_MODEL_OBJ is not None:
        try:
            SIGLIP_MODEL_OBJ.to("cpu")
        except Exception:
            pass

    SIGLIP_MODEL_OBJ = None
    SIGLIP_PROCESSOR = None
    release_memory()
    gc.collect()


pending = [
    source for source in SOURCE_RECORDS
    if not completed(source) and source["document_id"] not in SKIP_DOCS
]

textbooks = [source for source in pending if source["category"] == "Textbooks"]

if textbooks:
    batch = textbooks[:TEXTBOOKS_PER_RUN]
else:
    batch, pages = [], 0

    for source in pending:
        n = page_count(source)

        if batch and (pages + n > OTHER_PAGE_BUDGET or len(batch) >= OTHER_MAX_DOCS):
            break

        batch.append(source)
        pages += n

        if pages >= OTHER_PAGE_BUDGET:
            break


print(f"Completed : {sum(completed(x) for x in SOURCE_RECORDS)}/{len(SOURCE_RECORDS)}")
print(f"Pending   : {len(pending)}")
print(f"This run  : {len(batch)}")
print(f"Pages     : {sum(page_count(x) for x in batch)}")


BUILD_FAILURES = []

try:
    for i, source in enumerate(batch, 1):
        print(f"\n[{i}/{len(batch)}] {source['category']} — {source['filename']} ({page_count(source)} pages)")

        clean_retry(source)
        clear_document_memory()

        EMBEDDER.to(BUILD_DEVICE)
        release_memory()

        started = time.time()

        try:
            row = build_document(source)
            print(" built:", row["document_id"], "|", round(time.time() - started, 2), "s")
            del row

        except Exception as exc:
            failure = {
                "document_id": source["document_id"],
                "filename": source["filename"],
                "category": source["category"],
                "error": f"{type(exc).__name__}: {exc}",
            }
            BUILD_FAILURES.append(failure)
            print(" failed:", failure["error"])

        finally:
            clear_document_memory()
            print(" saved / memory cleared")

finally:
    run_mineru = _base_run_mineru
    get_siglip = _base_get_siglip
    build_visual_rag = _base_build_visual_rag
    ENCODE_DOCS = _base_encode_docs

    if _old_max_seq is not None:
        EMBEDDER.max_seq_length = _old_max_seq

    EMBEDDER.to(BUILD_DEVICE)
    release_memory()


BUILD_RESULTS = []

for source in SOURCE_RECORDS:
    path = DOCUMENTS_ROOT / source["document_id"] / "manifest.json"

    if not path.is_file():
        continue

    manifest = json.loads(path.read_text(encoding="utf-8"))

    BUILD_RESULTS.append({
        "status": "built",
        "document_id": manifest["document_id"],
        "category": manifest.get("category"),
        "title": manifest.get("title"),
        "source_path": manifest.get("source", {}).get("source_path"),
        "router_text": manifest.get("router_text", ""),
        "counts": manifest.get("counts", {}),
    })


write_jsonl(CORPUS_ROOT / "build_results.jsonl", BUILD_RESULTS)
write_jsonl(CORPUS_ROOT / "build_failures.jsonl", BUILD_FAILURES)

print(f"\nTotal completed: {len(BUILD_RESULTS)}/{len(SOURCE_RECORDS)}")

Completed : 41/53
Pending   : 12
This run  : 12
Pages     : 278

[1/12] Landmark — Landmark_20.pdf (14 pages)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/26 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/26 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/26 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/10 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/10 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/10 [00:00<?, ?it/s]

 built: doc_90137160c572 | 64.89 s
 saved / memory cleared

Total completed: 53/53


# 11. Corpus Router

A compact global document index is built from each successful document's title, category, headings, and representative text. Queries first route to the most relevant documents before expensive four-modality retrieval begins.

In [15]:
BUILT_DOCUMENTS = []
for manifest_path in sorted(DOCUMENTS_ROOT.glob("*/manifest.json")):
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    BUILT_DOCUMENTS.append({
        "document_id": manifest["document_id"],
        "category": manifest.get("category"),
        "title": manifest.get("title") or manifest.get("source", {}).get("filename"),
        "source_path": manifest.get("source", {}).get("source_path"),
        "router_text": manifest.get("router_text", ""),
    })

DOC_RECORDS = []
for row_id, result in enumerate(BUILT_DOCUMENTS):
    DOC_RECORDS.append({"row_id": row_id, **result})

write_jsonl(CORPUS_ROOT / "documents.jsonl", DOC_RECORDS)
DOC_BY_ROW = {r["row_id"]: r for r in DOC_RECORDS}
DOC_BY_ID = {r["document_id"]: r for r in DOC_RECORDS}

if DOC_RECORDS:
    index_path = CORPUS_ROOT / "documents.faiss"
    doc_vectors = ENCODE_DOCS([r["router_text"] for r in DOC_RECORDS], batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True).astype("float32")
    doc_vectors = np.ascontiguousarray(doc_vectors); faiss.normalize_L2(doc_vectors)
    DOC_DENSE_INDEX = faiss.IndexFlatIP(doc_vectors.shape[1]); DOC_DENSE_INDEX.add(doc_vectors)
    faiss.write_index(DOC_DENSE_INDEX, str(index_path))

    DOC_BM25_TOKENIZER = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
    doc_tokens = DOC_BM25_TOKENIZER.tokenize([r["router_text"] for r in DOC_RECORDS], return_as="tuple")
    DOC_BM25 = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); DOC_BM25.index(doc_tokens)
else:
    DOC_DENSE_INDEX = None; DOC_BM25_TOKENIZER = None; DOC_BM25 = None

print("Router documents:", len(DOC_RECORDS))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/53 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/53 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/53 [00:00<?, ?it/s]

Router documents: 53


In [16]:
def route_documents(query, top_k=ROUTER_TOP_DOCS):
    if not DOC_RECORDS: return []
    k_dense = min(ROUTER_CFG["dense_k"], len(DOC_RECORDS))
    scores, rows = DOC_DENSE_INDEX.search(encode_query(query), k_dense)
    dense = [{"id": DOC_BY_ROW[int(row)]["document_id"], "rank": i, "score": float(score)}
             for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]

    qtok = DOC_BM25_TOKENIZER.tokenize([query], update_vocab=False, return_as="tuple")
    brow, bscores = DOC_BM25.retrieve(qtok, k=min(ROUTER_CFG["bm25_k"], len(DOC_RECORDS)), return_as="tuple", show_progress=False)
    lexical = [{"id": DOC_BY_ROW[int(row)]["document_id"], "score": float(score)} for row, score in zip(brow[0], bscores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(lexical, 1): hit["rank"] = i

    fused = weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, ROUTER_CFG["rrf_k"], ROUTER_CFG["candidate_k"])
    pool = fused[:min(len(fused), 10)]
    if pool:
        reranker = get_reranker()
        pairs = [(query, DOC_BY_ID[x["id"]]["router_text"]) for x in pool]
        scores = np.asarray(reranker.predict(pairs, prompt_name="research", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
        pool = [dict(item, rerank_score=float(score)) for item, score in zip(pool, scores)]
        pool.sort(key=lambda x: (-x["rerank_score"], x["rank"]))
    return [{**DOC_BY_ID[x["id"]], "router_score": x.get("rerank_score", x["score"])} for x in pool[:top_k]]

# 12. Lazy Document Runtime

Only routed documents are loaded into memory. FAISS indexes are restored from disk and compact BM25 indexes are reconstructed from the persisted records. A small in-memory LRU pool prevents all document indexes from occupying RAM at once.

In [17]:
RUNTIME_POOL = OrderedDict()


def build_bm25_runtime(texts):
    if not texts: return None, None
    tok = Tokenizer(splitter=technical_split, stopwords=[], stemmer=None)
    tokens = tok.tokenize(texts, return_as="tuple")
    bm = bm25s.BM25(method="lucene", k1=1.2, b=0.75, backend="numpy"); bm.index(tokens)
    return bm, tok


def load_document_runtime(document_id):
    if document_id in RUNTIME_POOL:
        runtime = RUNTIME_POOL.pop(document_id); RUNTIME_POOL[document_id] = runtime
        return runtime

    root = DOCUMENTS_ROOT / document_id
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    labels = json.loads((root / "canonical" / "labels.json").read_text(encoding="utf-8"))
    text_records = read_jsonl(root / "text" / "records.jsonl")
    math_cards = read_jsonl(root / "math" / "equation_cards.jsonl")
    math_edges = read_jsonl(root / "math" / "source_graph.jsonl")
    visuals = read_jsonl(root / "visual" / "records.jsonl")
    tables = read_jsonl(root / "table" / "tables.jsonl")
    table_rows = read_jsonl(root / "table" / "rows.jsonl")
    table_cells = read_jsonl(root / "table" / "cells.jsonl")

    text_bm, text_tok = build_bm25_runtime([r["bm25_text"] for r in text_records])
    math_bm, math_tok = build_bm25_runtime([r["lexical_text"] for r in math_cards])
    visual_text = ["\n".join(x for x in (r.get("label") or "", " > ".join(r.get("section_path") or []), r.get("source_text") or "", r.get("visible_text") or "") if x) for r in visuals]
    visual_bm, visual_tok = build_bm25_runtime(visual_text)
    table_bm, table_tok = build_bm25_runtime([r["table_text"] for r in tables])
    row_bm, row_tok = build_bm25_runtime([r["text"] for r in table_rows])

    runtime = {
        "document_id": document_id, "root": root, "manifest": manifest, "labels": labels,
        "text_records": text_records, "text_by_chunk": {r["chunk_id"]: r for r in text_records},
        "text_index": faiss.read_index(str(root / "text" / "dense.faiss")) if (root / "text" / "dense.faiss").is_file() else None,
        "text_bm": text_bm, "text_tok": text_tok,
        "math_cards": math_cards, "math_by_object": {r["object_id"]: r for r in math_cards},
        "math_by_label": {r["equation_label"].lower(): r for r in math_cards if r.get("equation_label")},
        "math_edges": math_edges, "math_index": faiss.read_index(str(root / "math" / "dense.faiss")) if (root / "math" / "dense.faiss").is_file() else None,
        "math_bm": math_bm, "math_tok": math_tok,
        "visuals": visuals, "visual_by_id": {r["visual_id"]: r for r in visuals},
        "visual_index": faiss.read_index(str(root / "visual" / "qwen.faiss")) if (root / "visual" / "qwen.faiss").is_file() else None,
        "visual_bm": visual_bm, "visual_tok": visual_tok,
        "visual_view_meta": read_jsonl(root / "visual" / "siglip_multiview_meta.jsonl"),
        "visual_view_vectors": np.load(root / "visual" / "siglip_multiview.npy") if (root / "visual" / "siglip_multiview.npy").is_file() else np.empty((0, 0), dtype="float32"),
        "tables": tables, "table_by_id": {r["table_id"]: r for r in tables}, "table_bm": table_bm, "table_tok": table_tok,
        "table_index": faiss.read_index(str(root / "table" / "tables.faiss")) if (root / "table" / "tables.faiss").is_file() else None,
        "table_rows": table_rows, "table_row_by_id": {r["row_id"]: r for r in table_rows}, "row_bm": row_bm, "row_tok": row_tok,
        "row_index": faiss.read_index(str(root / "table" / "rows.faiss")) if (root / "table" / "rows.faiss").is_file() else None,
        "table_cells": table_cells,
    }

    section_records = defaultdict(list)
    for r in text_records: section_records[tuple(r["section_path"])].append(r)
    runtime["section_records"] = section_records
    runtime["section_position"] = {r["chunk_id"]: (section, i) for section, rows in section_records.items() for i, r in enumerate(rows)}

    ref_index = defaultdict(list)
    for r in visuals:
        if r.get("reference_key"): ref_index[r["reference_key"]].append(r["visual_id"])
    runtime["visual_reference_index"] = ref_index

    RUNTIME_POOL[document_id] = runtime
    while len(RUNTIME_POOL) > MAX_RUNTIME_DOCS:
        RUNTIME_POOL.popitem(last=False)
    return runtime

# 13. Query Planner

The planner selects modalities from the user query. Corpus routing chooses documents separately, so references such as page numbers are interpreted only inside the routed documents.

In [18]:
NUMERIC_OPERATION_RE = re.compile(r"\b(?:average|mean|sum|total|percentage|percent|difference|ratio|calculate|compute)\b", re.I)
FIGURE_WORD_RE = re.compile(r"\b(?:figure|fig\.?|diagram|plot|graph|chart|architecture|image|visual)\b", re.I)
TABLE_WORD_RE = re.compile(r"\b(?:table|row|column|cell|coverage)\b", re.I)
MATH_WORD_RE = re.compile(r"\b(?:eq(?:uation)?s?\.?|formula|mathematical expression|derive|derivative|jacobian|softmax|sigmoid|logits?)\b", re.I)
TEXT_WORD_RE = re.compile(r"\b(?:why|explain|describe|discuss|what|how|meaning|represent|compare|summarize)\b", re.I)
PAGE_REFERENCE_RE = re.compile(r"\b(?:pdf\s*)?page\s*(\d+)\b", re.I)
EXPLICIT_EQ_RE = re.compile(r"\b(?:eq(?:uation)?s?\.?|formula)\s*\(?\s*([A-Za-z]\.?\d+|\d+(?:\.\d+)+)\s*\)?", re.I)
EXACT_VISUAL_RE = re.compile(r"\b(?:figure|fig\.?|chart)\s*[:#]?\s*(\d+(?:\.\d+)*[a-z]?)\b", re.I)
TABLE_REFERENCE_RE = re.compile(r"\btable\s*(\d+(?:\.\d+)*)", re.I)


def detect_modalities(query):
    mods = []
    if TABLE_WORD_RE.search(query): mods.append("table")
    if FIGURE_WORD_RE.search(query): mods.append("visual")
    if MATH_WORD_RE.search(query): mods.append("math")
    if TEXT_WORD_RE.search(query) or not mods: mods.append("text")
    return list(dict.fromkeys(mods))


def plan_query(query):
    query = clean_text(query)
    if not query: return {"status": "invalid_query", "query": query, "modalities": []}
    modalities = detect_modalities(query)
    if NUMERIC_OPERATION_RE.search(query) and TABLE_WORD_RE.search(query):
        return {"status": "numeric_engine_required", "query": query, "modalities": ["table"]}
    return {"status": "plan_ready", "query": query, "modalities": modalities, "pdf_pages": [int(x) for x in PAGE_REFERENCE_RE.findall(query)]}

# 14. Per-Document Retrieval

The same Text, Math, Visual, and Table strategy used in the final demo is applied to each routed document. Evidence remains source-bound and carries document identity and page provenance.

In [19]:
def dense_hits(index, records, query, id_field, k):
    if index is None or not records: return []
    scores, rows = index.search(encode_query(query), min(k, len(records)))
    return [{"id": records[int(row)][id_field], "rank": i, "score": float(score), "record": records[int(row)]}
            for i, (row, score) in enumerate(zip(rows[0], scores[0]), 1) if row >= 0]


def bm25_hits(bm, tok, records, query, id_field, k):
    if bm is None or tok is None or not records: return []
    qtok = tok.tokenize([query], update_vocab=False, return_as="tuple")
    rows, scores = bm.retrieve(qtok, k=min(k, len(records)), return_as="tuple", show_progress=False)
    hits = [{"id": records[int(row)][id_field], "score": float(score), "record": records[int(row)]}
            for row, score in zip(rows[0], scores[0]) if row >= 0 and float(score) > 0]
    for i, hit in enumerate(hits, 1): hit["rank"] = i
    return hits


def bind_evidence(runtime, item, score=0.0):
    source = runtime["manifest"]["source"]
    out = dict(item)
    out["document_id"] = runtime["document_id"]
    out["source_title"] = runtime["manifest"].get("title") or source.get("filename")
    out["source_category"] = runtime["manifest"].get("category")
    out["source_pdf_sha256"] = source["source_sha256"]
    out["retrieval_score"] = float(score)
    out["evidence_id"] = f"{runtime['document_id']}:{item['modality']}:{stable_hash(item['source_object_id'], item['pdf_page_number'], n=12)}"
    return out

In [20]:
def retrieve_text_doc(runtime, query, top_k=5):
    dense = dense_hits(runtime["text_index"], runtime["text_records"], query, "chunk_id", TEXT_CFG["dense_k"])
    lexical = bm25_hits(runtime["text_bm"], runtime["text_tok"], runtime["text_records"], query, "chunk_id", TEXT_CFG["bm25_k"])
    fused = weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, TEXT_CFG["rrf_k"], TEXT_CFG["rrf_top_k"])
    pool = {x["id"]: x["record"] for x in dense + lexical}
    candidates = [dict(x, record=pool[x["id"]]) for x in fused if x["id"] in pool]
    if not candidates: return []

    reranker = get_reranker()
    scores = np.asarray(reranker.predict([(query, c["record"]["dense_text"]) for c in candidates], prompt_name="research", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
    ranked = [dict(c, rerank_score=float(score)) for c, score in zip(candidates, scores)]
    ranked.sort(key=lambda x: (-x["rerank_score"], x["rank"]))
    anchors = ranked[:min(TEXT_CFG["anchor_budget"], len(ranked))]

    neighbors, anchor_ids = {}, {a["record"]["chunk_id"] for a in anchors}
    for anchor in anchors:
        position = runtime["section_position"].get(anchor["record"]["chunk_id"])
        if not position: continue
        section, i = position; rows = runtime["section_records"][section]
        for j in (i - 1, i + 1):
            if 0 <= j < len(rows) and rows[j]["chunk_id"] not in anchor_ids:
                neighbors[rows[j]["chunk_id"]] = rows[j]

    selected = [a["record"] for a in anchors] + list(neighbors.values())[:TEXT_CFG["neighbor_budget"]]
    seen, evidence = set(), []
    for row in selected:
        key = clean_text(unicodedata.normalize("NFKC", row["source_text"]))
        if key in seen: continue
        seen.add(key)
        item = {"source_object_id": row["chunk_id"], "pdf_page_number": row["page_start"] + 1, "modality": "text",
                "content": {"text": row["source_text"], "section_path": row["section_path"]},
                "provenance": {"source_object_ids": row["source_object_ids"], "page_range": [row["page_start"] + 1, row["page_end"] + 1]}}
        evidence.append(bind_evidence(runtime, item))
        if len(evidence) >= top_k: break
    return evidence

In [21]:
def requested_math_roles(query):
    q = clean_text(query).lower(); required, forbidden = set(), set()
    if "softmax" in q and any(x in q for x in ("define", "definition", "standalone", "expression")): required |= {"softmax", "standalone_softmax"}; forbidden |= {"loss"}
    if "softmax" in q and any(x in q for x in ("probability", "probabilities", "class logits", "normalized")): required.add("softmax")
    if "jacobian" in q and "determinant" in q: required.add("jacobian_determinant"); forbidden.add("jacobian_matrix")
    elif "jacobian" in q or "derivative" in q: required.add("jacobian_matrix"); forbidden.add("jacobian_determinant")
    if "inverse" in q and "mapping" in q: required.add("inverse_mapping"); forbidden.add("forward_mapping")
    elif "forward" in q and "mapping" in q: required.add("forward_mapping"); forbidden.add("inverse_mapping")
    if "attention" in q and "weight" in q: required.add("attention_weights")
    if "weighted sum" in q and "value" in q: required.add("weighted_value_sum")
    return required, forbidden


def retrieve_math_doc(runtime, query, top_k=3):
    explicit = [f"equation:{x}".lower() for x in EXPLICIT_EQ_RE.findall(query)]
    if explicit:
        cards = [runtime["math_by_label"][x] for x in explicit if x in runtime["math_by_label"]]
    else:
        dense = dense_hits(runtime["math_index"], runtime["math_cards"], query, "object_id", MATH_CFG["dense_k"])
        lexical = bm25_hits(runtime["math_bm"], runtime["math_tok"], runtime["math_cards"], query, "object_id", MATH_CFG["bm25_k"])
        fused = weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, MATH_CFG["rrf_k"], MATH_CFG["rerank_pool"])
        pool = {x["id"]: x["record"] for x in dense + lexical}
        candidates = [dict(x, card=pool[x["id"]]) for x in fused if x["id"] in pool]
        if candidates:
            reranker = get_reranker()
            pairs = [(query, "\n".join(["Section: " + " > ".join(c["card"]["section_path"]), "Context: " + " ".join(c["card"].get("source_contexts", [])[:2]), "Formula: " + strip_equation_tag(c["card"]["original_latex"]), "Roles: " + ", ".join(c["card"].get("formula_roles", []))])) for c in candidates]
            scores = np.asarray(reranker.predict(pairs, prompt_name="math_equation", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
            candidates = [dict(c, rerank_score=float(score)) for c, score in zip(candidates, scores)]
            candidates.sort(key=lambda x: (-x["rerank_score"], x["rank"]))
        required, forbidden = requested_math_roles(query)
        cards = []
        for item in candidates:
            roles = set(item["card"].get("formula_roles", []))
            if forbidden & roles: continue
            if required and not required.issubset(roles): continue
            cards.append(item["card"])
            if len(cards) >= top_k: break
        if not cards and not required and candidates: cards = [candidates[0]["card"]]

    evidence = []
    for card in cards[:top_k]:
        item = {"source_object_id": card["object_id"], "pdf_page_number": card["pdf_page_number"], "modality": "math",
                "content": {"latex": card["original_latex"], "equation_label": card.get("equation_label"), "section_path": card["section_path"]},
                "provenance": {"bbox_pdf": card.get("bbox_pdf"), "source_contexts": card.get("source_contexts", [])[:2]}}
        evidence.append(bind_evidence(runtime, item))
    return evidence

In [22]:
VISUAL_CUE_RE = re.compile(r"\b(diagram|architecture|plot|graph|curve|axis|legend|arrow|panel|image|picture|visual|dashed|dotted|blue|red|green|left|right|top|bottom)\b", re.I)


def encode_siglip_query(query):
    processor, model = get_siglip(SIGLIP_QUERY_DEVICE)
    inputs = processor(text=[query], padding="max_length", truncation=True, return_tensors="pt")
    text_inputs = {k: v.to(SIGLIP_QUERY_DEVICE) for k, v in inputs.items() if k in {"input_ids", "attention_mask"}}
    with torch.inference_mode():
        feats = model.get_text_features(**text_inputs)
        if hasattr(feats, "pooler_output"): feats = feats.pooler_output
        feats = torch.nn.functional.normalize(feats.float(), dim=-1)
    return feats.cpu().numpy()[0].astype("float32")


def retrieve_visual_doc(runtime, query, top_k=3):
    records = runtime["visuals"]
    if not records: return []
    exact = EXACT_VISUAL_RE.search(query)
    if exact:
        ids = runtime["visual_reference_index"].get(exact.group(1).lower(), [])
        ranked = [{"id": vid, "rank": i, "score": 1.0} for i, vid in enumerate(ids, 1)]
    else:
        dense = dense_hits(runtime["visual_index"], records, query, "visual_id", VISUAL_CFG["candidate_k"])
        lexical = bm25_hits(runtime["visual_bm"], runtime["visual_tok"], records, query, "visual_id", VISUAL_CFG["candidate_k"])
        lanes = {"qwen_source": dense, "bm25_source": lexical}
        weights = VISUAL_CFG["semantic_weights"]

        if VISUAL_CUE_RE.search(query) and runtime["visual_view_vectors"].size:
            q = encode_siglip_query(query); scores = runtime["visual_view_vectors"] @ q
            grouped = defaultdict(list)
            for i, meta in enumerate(runtime["visual_view_meta"]): grouped[meta["visual_id"]].append(i)
            rows = []
            for visual_id, idxs in grouped.items():
                best = max(idxs, key=lambda i: scores[i]); rows.append((visual_id, float(scores[best])))
            rows.sort(key=lambda x: (-x[1], x[0]))
            lanes["siglip_multiview"] = [{"id": vid, "rank": i, "score": score} for i, (vid, score) in enumerate(rows[:VISUAL_CFG["candidate_k"]], 1)]
            weights = VISUAL_CFG["visual_weights"]

        ranked = weighted_rrf(lanes, weights, VISUAL_CFG["rrf_k"], VISUAL_CFG["candidate_k"])
        pool = ranked[:VISUAL_CFG["rerank_pool"]]
        if pool:
            reranker = get_reranker()
            pairs = []
            for item in pool:
                row = runtime["visual_by_id"][item["id"]]
                text = "\n".join(x for x in (row.get("label") or "", row.get("caption") or "", row.get("source_text") or "", row.get("visible_text") or "") if x)
                pairs.append((query, text))
            scores = np.asarray(reranker.predict(pairs, prompt_name="research", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
            reranked = [dict(item, relevance_score=float(score)) for item, score in zip(pool, scores)]
            reranked.sort(key=lambda x: (-x["relevance_score"], x["rank"]))
            ids = {x["id"] for x in reranked}; ranked = reranked + [x for x in ranked if x["id"] not in ids]

    selected, used_groups = [], set()
    for item in ranked:
        row = runtime["visual_by_id"][item["id"]]; group = row.get("visual_group_id") or row["visual_id"]
        if group in used_groups: continue
        used_groups.add(group)
        evidence = {"source_object_id": row["visual_id"], "pdf_page_number": row["pdf_page_number"], "modality": "visual",
                    "content": {"label": row.get("label"), "caption": row.get("caption"), "visual_type": row["visual_type"], "crop_path": row["crop_path"], "visible_text": row.get("visible_text")},
                    "provenance": {"bbox_pdf": row["bbox_pdf"], "crop_sha256": row["crop_sha256"]}}
        selected.append(bind_evidence(runtime, evidence, item.get("relevance_score", item.get("score", 0.0))))
        if len(selected) >= top_k: break
    return selected

In [23]:
def hybrid_runtime_search(query, records, dense_index, bm, tok, id_field, k):
    dense = dense_hits(dense_index, records, query, id_field, k)
    lexical = bm25_hits(bm, tok, records, query, id_field, k)
    return weighted_rrf({"dense": dense, "bm25": lexical}, {"dense": 1.0, "bm25": 1.0}, TABLE_CFG["rrf_k"], k)


def retrieve_table_doc(runtime, query, top_k=3, page_constraint=None):
    tables = runtime["tables"]
    if not tables: return []
    explicit = TABLE_REFERENCE_RE.search(query)
    if explicit:
        target = f"table:{explicit.group(1)}".lower()
        selected_tables = [t for t in tables if (t.get("label") or "").lower() == target]
    elif page_constraint:
        selected_tables = [t for t in tables if t["pdf_page_number"] == page_constraint]
    else:
        ranked = hybrid_runtime_search(query, tables, runtime["table_index"], runtime["table_bm"], runtime["table_tok"], "table_id", TABLE_CFG["table_k"])
        selected_tables = [runtime["table_by_id"][x["id"]] for x in ranked]

    row_ranked = hybrid_runtime_search(query, runtime["table_rows"], runtime["row_index"], runtime["row_bm"], runtime["row_tok"], "row_id", TABLE_CFG["row_k"])
    row_by_table = defaultdict(list)
    for item in row_ranked:
        row = runtime["table_row_by_id"][item["id"]]; row_by_table[row["table_id"]].append(row)

    evidence = []
    for table in selected_tables:
        rows = row_by_table.get(table["table_id"], [])[:3]
        if not rows: rows = [runtime["table_row_by_id"][rid] for rid in table["row_ids"][:3] if rid in runtime["table_row_by_id"]]
        row_numbers = {r["row"] for r in rows}
        cells = [c for c in runtime["table_cells"] if c["table_id"] == table["table_id"] and c["row"] in row_numbers]
        item = {"source_object_id": table["table_id"], "pdf_page_number": table["pdf_page_number"], "modality": "table",
                "content": {"label": table.get("label"), "caption": table.get("caption"), "rows": rows,
                            "selected_cells": [{"cell_id": c["cell_id"], "row": c["row"], "col": c["col"], "header": c.get("header"), "raw_value": c["text"], "typed_value": c.get("typed_value")} for c in cells]},
                "provenance": {"bbox_pdf": table["bbox_pdf"], "reconstruction": table["reconstruction"]}}
        evidence.append(bind_evidence(runtime, item))
        if len(evidence) >= top_k: break
    return evidence

# 15. Cross-Document Retrieval

The query is routed to the best documents, each requested modality is searched inside those documents, duplicate evidence is removed, and one global reranker selects the strongest evidence across the corpus.

In [24]:
def evidence_text(item):
    content = item.get("content", {})
    if item["modality"] == "text": body = content.get("text", "")
    elif item["modality"] == "math": body = "\n".join([content.get("equation_label") or "", content.get("latex") or "", " ".join(item.get("provenance", {}).get("source_contexts", []))])
    elif item["modality"] == "visual": body = "\n".join(x for x in (content.get("label") or "", content.get("caption") or "", content.get("visible_text") or "") if x)
    elif item["modality"] == "table":
        rows = "\n".join(r.get("text", "") for r in content.get("rows", []))
        body = "\n".join(x for x in (content.get("label") or "", content.get("caption") or "", rows) if x)
    else: body = str(content)
    return f"Source: {item.get('source_title')}\nPage: {item.get('pdf_page_number')}\nModality: {item.get('modality')}\n{body}"


def global_rerank(query, evidence, top_k=TOP_FINAL_EVIDENCE):
    if not evidence: return []
    unique = {}
    for item in evidence:
        unique[(item["document_id"], item["modality"], item["source_object_id"], item["pdf_page_number"])] = item
    rows = list(unique.values())[:GLOBAL_RERANK_POOL]
    reranker = get_reranker()
    scores = np.asarray(reranker.predict([(query, evidence_text(x)) for x in rows], prompt_name="research", batch_size=4, show_progress_bar=False, convert_to_numpy=True)).reshape(-1)
    ranked = [dict(item, global_score=float(score)) for item, score in zip(rows, scores)]
    ranked.sort(key=lambda x: -x["global_score"])

    requested = detect_modalities(query)
    selected, used = [], set()
    for modality in requested:
        match = next((x for x in ranked if x["modality"] == modality and x["evidence_id"] not in used), None)
        if match:
            selected.append(match); used.add(match["evidence_id"])
    for item in ranked:
        if item["evidence_id"] in used: continue
        selected.append(item); used.add(item["evidence_id"])
        if len(selected) >= top_k: break
    return selected[:top_k]


def retrieve_corpus(query, top_docs=ROUTER_TOP_DOCS, top_evidence=TOP_FINAL_EVIDENCE):
    plan = plan_query(query)
    if plan["status"] != "plan_ready":
        return {"query": query, "status": plan["status"], "documents": [], "modalities": plan.get("modalities", []), "evidence": []}

    routed = route_documents(query, top_docs)
    collected = []
    for doc in routed:
        runtime = load_document_runtime(doc["document_id"])
        page_constraint = plan["pdf_pages"][0] if len(plan["pdf_pages"]) == 1 else None
        for modality in plan["modalities"]:
            if modality == "text": collected.extend(retrieve_text_doc(runtime, query))
            elif modality == "math": collected.extend(retrieve_math_doc(runtime, query))
            elif modality == "visual": collected.extend(retrieve_visual_doc(runtime, query))
            elif modality == "table": collected.extend(retrieve_table_doc(runtime, query, page_constraint=page_constraint))

    evidence = global_rerank(query, collected, top_evidence)
    return {"query": query, "status": "provisional_evidence" if evidence else "no_evidence",
            "documents": [{"document_id": x["document_id"], "title": x["title"], "category": x["category"], "router_score": x["router_score"]} for x in routed],
            "modalities": plan["modalities"], "evidence": evidence}

# 16. Ollama `llama3` Grounded Answer Layer

Only the final top evidence is sent to `llama3`. The prompt explicitly forbids unsupported claims and requires citations using evidence IDs such as `[E1]`, `[E2]`.

In [25]:
def evidence_for_llm(item, idx):
    content = item.get("content", {})
    base = {
        "evidence_id": f"E{idx}", "document_id": item["document_id"], "source_title": item.get("source_title"),
        "category": item.get("source_category"), "page": item.get("pdf_page_number"), "modality": item.get("modality"),
    }
    if item["modality"] == "text": base["content"] = short_text(content.get("text"), 1800)
    elif item["modality"] == "math":
        base["content"] = {"equation_label": content.get("equation_label"), "latex": content.get("latex"),
                           "source_context": [short_text(x, 500) for x in item.get("provenance", {}).get("source_contexts", [])[:2]]}
    elif item["modality"] == "visual":
        base["content"] = {"label": content.get("label"), "caption": short_text(content.get("caption"), 900), "visible_text": short_text(content.get("visible_text"), 600)}
    elif item["modality"] == "table":
        base["content"] = {"label": content.get("label"), "caption": short_text(content.get("caption"), 500),
                           "rows": [short_text(r.get("text"), 400) for r in content.get("rows", [])[:4]],
                           "cells": content.get("selected_cells", [])[:16]}
    return base


SYSTEM_PROMPT = """You are RAGScholar, a source-grounded academic assistant.
Use ONLY the evidence supplied in the user message. Do not use outside knowledge to add unsupported claims.
If the evidence is incomplete, say that clearly. If sources disagree, describe the disagreement.
Preserve equations exactly when citing them. Visual evidence contains caption/OCR/context, not full visual reasoning, so do not invent unseen image details.
Cite factual claims inline using [E1], [E2], etc. Return JSON with keys: answer, citations, confidence, status.
confidence must be one of: high, medium, low. status must be answered or insufficient_evidence.
"""


def llama3_answer(query, evidence):
    if not evidence:
        return {"answer": "I could not find enough source-grounded evidence to answer this question.", "citations": [], "confidence": "low", "status": "insufficient_evidence"}

    packet = [evidence_for_llm(item, i) for i, item in enumerate(evidence, 1)]
    user_prompt = "QUESTION:\n" + query + "\n\nEVIDENCE:\n" + json.dumps(packet, ensure_ascii=False, indent=2)
    payload = {
        "model": OLLAMA_MODEL,
        "stream": False,
        "format": "json",
        "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}],
        "options": {"temperature": 0.1, "num_ctx": 8192},
    }
    payload["keep_alive"] = OLLAMA_KEEP_ALIVE

    moved = False
    if OLLAMA_GPU_FRIENDLY and DEVICE == "cuda":
        EMBEDDER.to("cpu")
        if RERANKER is not None: RERANKER.model.to("cpu")
        if SIGLIP_MODEL_OBJ is not None: SIGLIP_MODEL_OBJ.to("cpu")
        release_memory(); moved = True

    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600)
        response.raise_for_status()
        content = response.json()["message"]["content"]
    finally:
        if moved:
            EMBEDDER.to(DEVICE)
            if RERANKER is not None: RERANKER.model.to(DEVICE)
            release_memory()

    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        result = {"answer": content, "citations": [], "confidence": "medium", "status": "answered"}
    return result

# 17. Public RAG API

`ask()` is the final user-facing function. It performs corpus routing, four-modality retrieval, top-10 global evidence selection, and `llama3` synthesis in one call.

In [26]:
def ask(query, top_docs=ROUTER_TOP_DOCS, top_evidence=TOP_FINAL_EVIDENCE):
    retrieval = retrieve_corpus(query, top_docs=top_docs, top_evidence=top_evidence)
    if retrieval["status"] != "provisional_evidence":
        return {"query": query, "status": retrieval["status"], "answer": None, "citations": [], "confidence": "low",
                "documents": retrieval["documents"], "modalities": retrieval["modalities"], "evidence": retrieval["evidence"]}

    generated = llama3_answer(query, retrieval["evidence"])
    return {
        "query": query, "status": generated.get("status", "answered"), "answer": generated.get("answer", ""),
        "citations": generated.get("citations", []), "confidence": generated.get("confidence", "medium"),
        "documents": retrieval["documents"], "modalities": retrieval["modalities"], "evidence": retrieval["evidence"],
    }

# 18. Corpus Manifest

The corpus manifest records the successful document builds, failures, models, and the final public API.

In [27]:
CORPUS_MANIFEST = {
    "schema": "ragscholar.full_corpus_manifest_v2",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "documents_discovered": len(SOURCE_RECORDS), "documents_built": len(DOC_RECORDS), "documents_failed": len(BUILD_FAILURES),
    "models": {"embedding": EMBED_MODEL, "reranker": RERANK_MODEL, "siglip": SIGLIP_MODEL, "answer_model": OLLAMA_MODEL},
    "retrieval": {"modalities": ["text", "math", "visual", "table"], "top_documents": ROUTER_TOP_DOCS, "top_final_evidence": TOP_FINAL_EVIDENCE,
                  "retrieval_api": "retrieve_corpus", "answer_api": "ask"},
    "environment": {"python": sys.version.split()[0], "torch": torch.__version__, "pymupdf": package_version("PyMuPDF"),
                    "faiss": package_version("faiss-cpu") or package_version("faiss-gpu"), "bm25s": package_version("bm25s"),
                    "sentence_transformers": package_version("sentence-transformers"), "transformers": package_version("transformers")},
}
write_json(CORPUS_ROOT / "manifest.json", CORPUS_MANIFEST)
print("Corpus manifest:", CORPUS_ROOT / "manifest.json")

Corpus manifest: C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\artifacts\ragscholar_full\corpus\manifest.json


# 19. Example Queries

Run these after the full corpus build and with Ollama running locally. `llama3` must be available under that exact Ollama model name.

In [28]:
DEMO_QUERIES = [
    "Explain why dropout is less effective in convolutional layers.",
    "Which equation defines the standalone softmax function?",
    "Find and explain the transformer architecture figure.",
    "Compare how the corpus discusses GANs and VAEs.",
]

# Example:
# result = ask(DEMO_QUERIES[0])
# display(Markdown(result["answer"]))
# print("Citations:", result["citations"])
# display(pd.DataFrame([{k: e.get(k) for k in ("evidence_id", "source_title", "modality", "pdf_page_number", "global_score")} for e in result["evidence"]]))

---
## End of RAGScholar Full Corpus Notebook

The complete corpus now follows one shared architecture: per-document four-modality indexing, corpus-level routing, global evidence reranking, top-10 source-grounded context, and local Ollama `llama3` answer synthesis.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.app import start_app

WEB_SERVER = start_app(
    retrieve_fn=retrieve_corpus,
    documents_root=DOCUMENTS_ROOT,
    corpus_count=len(SOURCE_RECORDS),
    project_root=PROJECT_ROOT,
    port=5000,
)

RAGScholar: http://127.0.0.1:5000


127.0.0.1 - - [06/Oct/2026 20:33:14] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:33:14] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:33:14] "GET /static/app.js HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:33:15] "GET /api/health HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:33:15] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [06/Oct/2026 20:33:15] "GET /api/conversations HTTP/1.1" 200 -


Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Default prompt name is set to 'research'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


...

BM25S Count Tokens:   0%|          | 0/162 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/162 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenize texts:   0%|          | 0/1 [00:00<?, ?it/s]

127.0.0.1 - - [06/Oct/2026 20:39:41] "POST /api/chat HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:39:41] "GET /api/conversations HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:40:19] "GET /api/page/doc_1e0651b6810e/5 HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:42:18] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:42:18] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [06/Oct/2026 20:42:18] "GET /static/app.js HTTP/1.1" 304 -
127.0.0.1 - - [06/Oct/2026 20:42:18] "GET /api/health HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:42:18] "GET /api/conversations HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:42:31] "GET /api/conversations/7efade09f3fc4379a530d1b0d354a66a HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 20:42:31] "GET /E2 HTTP/1.1" 404 -
127.0.0.1 - - [06/Oct/2026 22:01:27] "GET /api/conversations/7efade09f3fc4379a530d1b0d354a66a HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 22:01:27] "GET /E2 HTTP/1.1" 404 -


In [31]:
from app.app import start_app

start_app(
    retrieve_fn=retrieve_corpus,
    documents_root=DOCUMENTS_ROOT,
    host="127.0.0.1",
    port=5000,
    open_browser=True,
)

RAGScholar: http://127.0.0.1:5000


<ServerThread(Thread-57, started daemon 32352)>

127.0.0.1 - - [06/Oct/2026 23:48:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [06/Oct/2026 23:48:32] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [06/Oct/2026 23:48:32] "GET /static/app.js HTTP/1.1" 304 -
